<a href="https://colab.research.google.com/github/samuelaojih/Google-Colab/blob/main/Landsat_SLC_Correction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Install required packages
!pip install --upgrade xee
!pip install -U geemap
!pip install scipy rioxarray rasterio

In [ ]:
# Import libraries
import ee
import geemap
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
from scipy import interpolate
from scipy.ndimage import uniform_filter, gaussian_filter
import rioxarray
import rasterio
from rasterio.transform import from_bounds
from rasterio.crs import CRS
import zipfile
from pathlib import Path
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Initialize Earth Engine
ee.Authenticate()
ee.Initialize(
    project = 'ee-samuelcool28',
    opt_url = 'https://earthengine-highvolume.googleapis.com'
)

In [ ]:
# ============================================
# LOAD SHAPEFILE ASSET FROM GEE
# ============================================

# Load your shapefile asset
shapefile_asset = "projects/ee-samuelcool28/assets/Bauchi_lga"
roi = ee.FeatureCollection(shapefile_asset).geometry()

# Get ROI bounds for proper georeferencing
roi_bounds = roi.bounds().getInfo()['coordinates'][0]
min_x = min(coord[0] for coord in roi_bounds)
max_x = max(coord[0] for coord in roi_bounds)
min_y = min(coord[1] for coord in roi_bounds)
max_y = max(coord[1] for coord in roi_bounds)

print("✓ Shapefile loaded successfully")
print(f"✓ ROI bounds: Lon [{min_x:.4f}, {max_x:.4f}], Lat [{min_y:.4f}, {max_y:.4f}]")

# Create map and add ROI for visualization
map = geemap.Map(basemap='SATELLITE')
map.add_layer(ee.Image().paint(roi, 0, 2), {'palette': 'red'}, 'ROI')
map.centerObject(roi, 10)
map

In [ ]:
# ============================================
# FUNCTION: ADD NDVI WITH PROPER MASKING
# ============================================

def add_ndvi_with_mask(img):
    """
    Calculate NDVI with proper masking for clouds AND SLC-off gaps
    """
    # QA_PIXEL band for cloud detection
    qa_band = img.select('QA_PIXEL')

    # Cloud bits
    bit1 = qa_band.bitwiseAnd(1 << 1).neq(0)  # Dilated cloud
    bit3 = qa_band.bitwiseAnd(1 << 3).neq(0)  # Cloud shadow
    bit4 = qa_band.bitwiseAnd(1 << 4).neq(0)  # Cloud

    # QA_RADSAT band for SLC-off gaps
    radsat = img.select('QA_RADSAT')
    slc_gap = radsat.eq(1)  # Saturation/No data indicates gaps

    # Combine masks
    cloud_mask = bit1.Or(bit3).Or(bit4)
    invalid_mask = cloud_mask.Or(slc_gap)

    # Scale surface reflectance
    sr = img.select('SR_B.*').multiply(2.75e-05).add(-0.2)

    # Calculate NDVI (B4 = NIR, B3 = Red)
    ndvi = sr.normalizedDifference(['SR_B4', 'SR_B3']).rename('ndvi')

    # Apply mask
    return ndvi.updateMask(invalid_mask.Not()).copyProperties(img, ['system:time_start'])

In [ ]:
# ============================================
# LOAD LANDSAT 7 COLLECTION
# ============================================

print("\nLoading Landsat 7 collection...")
landsat7 = (
    ee.ImageCollection("LANDSAT/LE07/C02/T1_L2")
    .filterDate('2004-01-01', '2004-05-31')
    .filterBounds(roi)
    .filter(ee.Filter.lt('CLOUD_COVER', 10))
    .map(add_ndvi_with_mask)
)

image_count = landsat7.size().getInfo()
print(f"✓ Found {image_count} images within Bauchi LGA")

if image_count == 0:
    print("⚠️ No images found. Adjusting date range...")
    landsat7 = (
        ee.ImageCollection("LANDSAT/LE07/C02/T1_L2")
        .filterDate('2004-01-01', '2004-12-31')
        .filterBounds(roi)
        .filter(ee.Filter.lt('CLOUD_COVER', 10))
        .map(add_ndvi_with_mask)
    )
    image_count = landsat7.size().getInfo()
    print(f"✓ Found {image_count} images with expanded date range")

In [ ]:
# ============================================
# CONVERT TO XARRAY WITH PROPER COORDINATES
# ============================================

print("\nConverting to xarray...")
# Get projection info from first image
first_image = landsat7.first()
projection = first_image.select('ndvi').projection().getInfo()
crs = projection['crs']
transform = projection['transform']

print(f"✓ Native CRS: {crs}")

# Convert to xarray with original projection
ds = xr.open_dataset(
    landsat7,
    engine='ee',
    crs=crs,  # Use native CRS instead of EPSG:3857
    scale=30,  # Landsat native resolution is 30m
    geometry=roi
)

# Sort by time
ds = ds.sortby('time') * 1
print(f"✓ Dataset shape: {ds.ndvi.shape}")
print(f"✓ Time range: {ds.time.values[0]} to {ds.time.values[-1]}")

# Extract coordinate info for georeferencing
x_coords = ds.X.values
y_coords = ds.Y.values
pixel_width = abs(x_coords[1] - x_coords[0])
pixel_height = abs(y_coords[1] - y_coords[0])

print(f"✓ Pixel size: {pixel_width:.2f} m x {pixel_height:.2f} m")
print(f"✓ X range: [{x_coords.min():.2f}, {x_coords.max():.2f}]")
print(f"✓ Y range: [{y_coords.min():.2f}, {y_coords.max():.2f}]")

In [ ]:
# ============================================
# FUNCTION: FILL SCAN LINES (2D INTERPOLATION)
# ============================================

def fill_scanlines_2d(da):
    """
    Fill scan line gaps using 2D interpolation
    """
    data = da.values.copy()
    mask = np.isnan(data)

    if not mask.any():
        return da

    # Get coordinates of valid points
    y_valid, x_valid = np.where(~mask)
    values_valid = data[~mask]

    # Create grid of all points
    y_grid, x_grid = np.mgrid[0:data.shape[0], 0:data.shape[1]]

    try:
        # Linear interpolation
        interp = interpolate.griddata(
            (y_valid, x_valid),
            values_valid,
            (y_grid, x_grid),
            method='linear',
            fill_value=np.nan
        )

        # Fill remaining NaNs with nearest neighbor
        if np.isnan(interp).any():
            interp_nearest = interpolate.griddata(
                (y_valid, x_valid),
                values_valid,
                (y_grid, x_grid),
                method='nearest'
            )
            nan_mask = np.isnan(interp)
            interp[nan_mask] = interp_nearest[nan_mask]

        # Fill only the original gaps
        filled_data = data.copy()
        filled_data[mask] = interp[mask]

        # Apply light smoothing
        filled_data = uniform_filter(filled_data, size=3)

        return xr.DataArray(filled_data, dims=da.dims, coords=da.coords)

    except:
        # Fallback: row-wise interpolation
        return fill_scanlines_rowwise(da)

def fill_scanlines_rowwise(da):
    """Fallback: Fill gaps row by row"""
    data = da.values.copy()

    for i in range(data.shape[0]):
        row = data[i, :]
        mask_row = np.isnan(row)

        if mask_row.all():
            if i > 0 and i < data.shape[0] - 1:
                data[i, :] = (data[i-1, :] + data[i+1, :]) / 2
            elif i > 0:
                data[i, :] = data[i-1, :]
            else:
                data[i, :] = data[i+1, :]

        elif mask_row.any():
            x = np.arange(len(row))
            valid_x = x[~mask_row]
            valid_vals = row[~mask_row]

            if len(valid_x) > 1:
                row[mask_row] = np.interp(x[mask_row], valid_x, valid_vals)
                data[i, :] = row

    return xr.DataArray(data, dims=da.dims, coords=da.coords)

In [ ]:
# ============================================
# APPLY GAP FILLING
# ============================================

print("\nFilling scan lines...")
filled_images = []
for i in range(len(ds.time)):
    if i % 10 == 0:
        print(f"  Processing image {i+1}/{len(ds.time)}")
    filled = fill_scanlines_2d(ds.ndvi.isel(time=i))
    filled_images.append(filled)

filled_ndvi = xr.concat(filled_images, dim='time')
filled_ndvi = filled_ndvi.assign_coords(time=ds.time)
print("✓ Gap filling complete")

In [ ]:
# ============================================
# TEMPORAL AND SPATIAL SMOOTHING
# ============================================

print("\nApplying temporal smoothing...")
ndvi_temporal = filled_ndvi.rolling(time=2, min_periods=1, center=True).mean()

print("Applying spatial smoothing...")
def apply_gaussian_smoothing(da, sigma=0.5):
    smoothed = np.zeros_like(da.values)
    for i in range(len(da.time)):
        smoothed[i] = gaussian_filter(da.values[i], sigma=sigma)
    return xr.DataArray(smoothed, dims=da.dims, coords=da.coords)

ndvi_final = apply_gaussian_smoothing(ndvi_temporal, sigma=0.5)
ndvi_final = ndvi_final.assign_coords(time=ds.time)
print("✓ Smoothing complete")

In [ ]:
# ============================================
# FUNCTION: EXPORT PROPERLY GEOREFERENCED GEOTIFF
# ============================================

def export_georeferenced_tiff(da, filepath, crs_string, x_coords, y_coords):
    """
    Export a DataArray as a properly georeferenced GeoTIFF
    """
    # Get the data
    data = da.values

    # Handle NaN values (set to -9999 for NoData)
    data_nan = np.nan_to_num(data, nan=-9999)

    # Calculate geotransform
    x_min = x_coords.min()
    x_max = x_coords.max()
    y_min = y_coords.min()
    y_max = y_coords.max()

    pixel_width = (x_max - x_min) / data.shape[1]
    pixel_height = (y_max - y_min) / data.shape[0]

    transform = from_bounds(x_min, y_min, x_max, y_max, data.shape[1], data.shape[0])

    # Define CRS (handle different formats)
    if crs_string.startswith('EPSG:'):
        crs = CRS.from_string(crs_string)
    else:
        crs = CRS.from_string(crs_string)

    # Write GeoTIFF
    with rasterio.open(
        filepath,
        'w',
        driver='GTiff',
        height=data.shape[0],
        width=data.shape[1],
        count=1,
        dtype=data_nan.dtype,
        crs=crs,
        transform=transform,
        nodata=-9999,
        compress='lzw'
    ) as dst:
        dst.write(data_nan, 1)
        # Add metadata
        dst.update_tags(NDVI='Landsat7_Corrected', Units='NDVI')

    return filepath


In [ ]:
# ============================================
# EXPORT AS PROPERLY GEOREFERENCED GEOTIFF
# ============================================

print("\n" + "="*50)
print("EXPORTING GEOREFERENCED GEOTIFF FILES")
print("="*50)

# Create export directory
export_dir = Path("Bauchi_lga_landsat7_corrected")
export_dir.mkdir(exist_ok=True)

# Get CRS from the dataset (should be the native Landsat CRS)
# Typically UTM zone for Nigeria
crs_string = f"EPSG:{projection['crs']['epsg'] if 'epsg' in projection['crs'] else 32632}"  # Default to UTM 32N for Nigeria

print(f"✓ Using CRS: {crs_string}")

exported_files = []
date_strings = ndvi_final.time.dt.strftime('%Y%m%d').values

# Also create a world file (.tfw) for backup
for i, date_str in enumerate(date_strings):
    filename = f"Bauchi_lga_ndvi_{date_str}.tif"
    filepath = export_dir / filename

    # Export with proper georeferencing
    export_georeferenced_tiff(
        ndvi_final.isel(time=i),
        filepath,
        crs_string,
        ndvi_final.X.values,
        ndvi_final.Y.values
    )

    exported_files.append(filepath)

    if (i + 1) % 10 == 0 or (i + 1) == len(date_strings):
        print(f"  Exported {i+1}/{len(date_strings)} images")

# Also create a PRJ file for each (optional but helpful)
for filepath in exported_files:
    prj_file = filepath.with_suffix('.prj')
    with open(prj_file, 'w') as f:
        f.write(crs_string)

# Create ZIP file
zip_filename = f"Bauchi_lga_landsat7_corrected_{datetime.now().strftime('%Y%m%d_%H%M%S')}.zip"
print(f"\nCreating ZIP file: {zip_filename}")

with zipfile.ZipFile(zip_filename, 'w', zipfile.ZIP_DEFLATED) as zipf:
    for filepath in exported_files:
        zipf.write(filepath, filepath.name)
        # Include PRJ file
        prj_file = filepath.with_suffix('.prj')
        if prj_file.exists():
            zipf.write(prj_file, prj_file.name)
            prj_file.unlink()
        filepath.unlink()  # Delete individual file

# Remove empty directory
export_dir.rmdir()

# Get file size
file_size_mb = Path(zip_filename).stat().st_size / 1024 / 1024

print(f"\n✓ Export complete!")
print(f"  ZIP file: {zip_filename}")
print(f"  Size: {file_size_mb:.1f} MB")
print(f"  Images: {len(exported_files)} GeoTIFF files")
print(f"  CRS: {crs_string}")
print(f"  Pixel size: {pixel_width:.2f} m")

In [ ]:
# ============================================
# CREATE AUXILIARY FILES FOR ARC GIS
# ============================================

# Create a README file
readme_content = f"""
========================================
LANDSAT 7 CORRECTED NDVI IMAGES
========================================

Location: Bauchi LGA, Nigeria
Date range: {date_strings[0]} to {date_strings[-1]}
Number of images: {len(exported_files)}
CRS: {crs_string}
Pixel size: {pixel_width:.2f} m
NDVI range: -1 to 1

FILES INCLUDED:
- {len(exported_files)} GeoTIFF files with corrected NDVI values
- PRJ files with projection information

HOW TO USE IN ArcGIS:
1. Extract the ZIP file
2. Add the GeoTIFF files to ArcMap/ArcGIS Pro
3. They should automatically align to their correct地理位置
4. If not, check that the PRJ files are in the same directory

PROCESSING DETAILS:
- Scan lines removed using 2D interpolation
- Temporal smoothing applied (4-image rolling window)
- Spatial smoothing applied (Gaussian filter, sigma=0.5)
- Cloud masking applied

CONTACT: Generated from Google Earth Engine
========================================
"""

readme_file = Path("README_Bauchi_LGA.txt")
with open(readme_file, 'w') as f:
    f.write(readme_content)

# Add README to ZIP
with zipfile.ZipFile(zip_filename, 'a', zipfile.ZIP_DEFLATED) as zipf:
    zipf.write(readme_file, readme_file.name)

readme_file.unlink()

print(f"✓ Added README file to ZIP")

In [ ]:
# ============================================
# DOWNLOAD ZIP (FOR COLAB/JUPYTER)
# ============================================

try:
    from google.colab import files
    print("\nDownloading ZIP file...")
    files.download(zip_filename)
    print("✓ Download started")
except:
    print(f"\nZIP file saved as: {zip_filename}")
    print("You can download it manually from your file browser")

In [ ]:
# ============================================
# VERIFICATION - TEST ONE FILE
# ============================================

print("\n" + "="*50)
print("VERIFICATION")
print("="*50)

# Extract and verify one file
import tempfile
with tempfile.TemporaryDirectory() as tmpdir:
    test_file = Path(tmpdir) / "test.tif"

    # Export a test file
    export_georeferenced_tiff(
        ndvi_final.isel(time=0),
        test_file,
        crs_string,
        ndvi_final.X.values,
        ndvi_final.Y.values
    )

    # Read back and verify
    with rasterio.open(test_file) as src:
        print(f"✓ Test file CRS: {src.crs}")
        print(f"✓ Test file bounds: {src.bounds}")
        print(f"✓ Test file transform: {src.transform}")
        print(f"✓ Test file size: {src.width} x {src.height}")
        print(f"✓ NoData value: {src.nodata}")

print("\n" + "="*50)
print("✅ ALL TASKS COMPLETED SUCCESSFULLY!")
print("="*50)
print(f"Location: Bauchi LGA, Nigeria")
print(f"Images processed: {len(exported_files)}")
print(f"Output file: {zip_filename}")
print(f"File size: {file_size_mb:.1f} MB")
print("\nIMPORTANT: The GeoTIFF files are now properly georeferenced")
print("and will open in the correct location in ArcGIS!")
print("="*50)